<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T10-L04-metrics-that-punish-correct-answers/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T10-L04-metrics-that-punish-correct-answers/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T10-L04-metrics-that-punish-correct-answers/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T10-L04-metrics-that-punish-correct-answers/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T10-L04 · Metrics that punish correct answers: edit distance, BLEU and chrF against what a reader would accept

**You will build:** three string-similarity metrics from scratch (edit distance, BLEU and chrF),
a measurement of how often each one ranks an answer a reader would REJECT above an answer a
reader would ACCEPT, a text normaliser, and the comparison that shows what normalising fixes,
what it cannot, and what a rubric check that looks at quantities and units adds.

**Time:** ~45 minutes · **Runs on:** a laptop CPU, no GPU, no download ·
**Prerequisites:** T00-L01-the-8gb-track (the habit of printing what you measured, and
never typing it).

There are no documents, no models and no network in this lesson. The answers are generated
from a small template, so for every answer we KNOW whether a reader would accept it, and each
metric can be scored against that. Real benchmarks do not give you that luxury, which is why
the scoring method has to be tested on a case where the truth is known.

By the end you will be able to:

1. Implement Levenshtein edit distance, and a similarity built from it, that work on strings
   and on lists of words.
2. Implement sentence-level BLEU (clipped n-gram precision, brevity penalty, geometric mean)
   and chrF (character n-gram F-score), and read what each one rewards.
3. Implement a pairwise misordering rate, and use it to measure, for each kind of harmless
   rewrite and each kind of real error, how often each metric ranks them the wrong way round.
4. Implement a normaliser for formatting differences, and measure how much of that error it
   removes and which kinds it leaves untouched.
5. Measure a quantity-and-unit rubric check on the error types it was written for and on
   error types it was not, and name the cases it cannot see.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T00-L01`

**This lesson assumes you can already:**

- write nested loops that fill in a table cell by cell from the cells already filled (the idea behind dynamic programming)
- get the n-grams of a list or a string with slicing, and count them with collections.Counter
- write a regular expression for re.sub, including a lookahead or a lookbehind
- explain precision and recall, and how an F-score combines them

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Speech and Language Processing (3rd ed. draft), chapter 2: Words and Tokens](https://web.stanford.edu/~jurafsky/slp3/2.pdf) · Dan Jurafsky and James H. Martin (Stanford; free draft) · *free* — section 2.9, minimum edit distance: the dynamic-programming table behind exercise 1; section 2.6, regular expressions including lookahead assertions, the tool exercise 5 leans on
- [Bleu: a Method for Automatic Evaluation of Machine Translation](https://aclanthology.org/P02-1040/) · Papineni, Roukos, Ward and Zhu (ACL 2002; free on the ACL Anthology) · *free* — section 2, the modified n-gram precision, brevity penalty and geometric mean that exercise 2 implements, in the authors' own words
- [chrF: character n-gram F-score for automatic MT evaluation](https://aclanthology.org/W15-3049/) · Maja Popovic (WMT 2015; free on the ACL Anthology) · *free* — the metric exercise 3 implements: character n-gram precision and recall, averaged, combined into an F-score
- [A Call for Clarity in Reporting BLEU Scores](https://aclanthology.org/W18-6319/) · Matt Post (WMT 2018; free on the ACL Anthology) · *free* — why the same BLEU can print different scores when the reference is tokenised or normalised differently, which is the problem exercise 5 works on
- [OmniDocBench: the README of the benchmark used as this lesson's worked example](https://github.com/opendatalab/OmniDocBench) · opendatalab (GitHub) · *free* — the news log and the metric list that section 1 reads, and the evaluation configuration if you want to see an item-level scorer in use

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import math
import os
import re
import signal
import sys
import threading
import time
import traceback
import unicodedata
from collections import Counter
from fractions import Fraction
from typing import Callable

import numpy as np

print("python", sys.version.split()[0], "· numpy", np.__version__, "· platform", sys.platform)

SEED = 20261002          # this lesson's one fixed seed (the date it was written): every
                          # generated answer derives from it, so re-running prints the same
                          # numbers on any machine.

_FAILED_CHECKS: list[str] = []
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("edit_distance",),
    "exercise 2": ("bleu",),
    "exercise 3": ("chrf",),
    "exercise 4": ("misorder_rate",),
    "exercise 5": ("normalise",),
}


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (chrf)"; several -> "exercises 2, 3 and 4"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


# A check that never returns would freeze the notebook (and the grader, which imports it). Every
# check below runs under a time limit far above what a correct answer needs (milliseconds), and
# so does every single call a demo makes into your code; the limits are short on purpose, because
# a loop that keeps adding to a list can use gigabytes in seconds; a whole demo gets a much longer limit, because slow but
# correct code is allowed to be slow.
_CHECK_SECONDS = 2
_CALL_SECONDS = 3
_DEMO_SECONDS = 90


class _OutOfTime(BaseException):
    """Raised inside a guarded block when its time is up. A BaseException, so an `except
    Exception:` in the code under test cannot swallow it, and it repeats every 0.25 s in case a
    bare `except:` does. The guard turns it into an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 2048):
    """Turn a call that never returns into a failed check that says so, instead of a notebook
    that hangs. It uses the operating system's alarm where there is one (Linux and macOS, in the
    main thread) and does nothing elsewhere. An alarm that whoever started us had already set
    keeps its own deadline if it is due first, and is put back, with the time it had left and
    its repeat interval, when the block ends. A correct answer finishes far inside the limit,
    and nothing that is printed depends on it."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped block's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        import gc
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: either a loop whose "
               "stopping condition is never met (a counter that is never advanced, an "
               "index that wraps round) or a computation that works out the same "
               "sub-problem over and over; cap every loop, and remember what you "
               "have already worked out")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_time_limit", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)

    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message, which names the likely mistake, and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        with _time_limit(_CHECK_SECONDS if label in _EXERCISES else _DEMO_SECONDS, label):
            check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. A worked example: a benchmark whose scoring was revised

OmniDocBench scores how well a model turns a document page into Markdown. Its text score is
an edit distance against one stored reference (tables and formulas have their own metrics,
which this lesson leaves alone). As of 2 October 2026 the project's README (its newest
commit is dated 11 September 2026) says, in its news log:

* on 10 April 2026 it went from v1.5 to v1.6, changing BOTH the data (new pages, fixed
  annotation typos) AND the evaluation code (a new matching procedure that keeps the ground
  truth fixed and adapts only the prediction's segmentation);
* on 30 April 2026 it went from v1.6 to v1.7 and added skills-based evaluation. The two
  later entries add a community integration and leaderboard rows, not a version;
* on 16 January 2025 it changed the edit distance for tables to use normalised ground truths
  and predictions: the same idea as this lesson's repair (the README says no more);
* its supported metrics include normalised edit distance and BLEU, its overall score includes
  one minus the text edit distance, and its Known Issues say unusual output formats cause
  matching failures that post-processing can reduce.

`claims.yaml` has the sources. The point is not the leaderboard: a score is only comparable
between two models if the same version of the scoring code produced both, and the entries above
show that code revised with the data (v1.6) and without it (January 2025). What does not
change is the mechanism this lesson teaches: when the reference is ONE string, a metric
compares strings, not meaning. So the lesson generates a small task of the same shape, with a
harmless rewrite and a real error planted on purpose, and measures the metrics on it.

## 2. The task we will generate

Each item is a sentence with a subject and two or three quantities with units, for example a
machine, its power, its mass and its length. The REFERENCE is one canonical rendering. Around
it the generator plants two families of candidate answers:

* **accepted**: a reader would accept these. Harmless Markdown or HTML markup, odd spacing
  and punctuation, letter case, the same facts in another order, a unit converted
  (kilowatts to watts, minutes to seconds), another wording of the same facts, and one that
  combines several of these;
* **rejected**: a reader would reject these. One digit changed, two values swapped between
  their attributes, a negation, a unit changed without converting the number, a fact left
  out, and the wrong subject.

A candidate is "accepted" when its quantities (units converted) and its subject are the
reference's, and nothing contradicts them. That is a planted truth standing in for a human
reader: it is exact for these templates and says nothing about borderline judgements, which
real answers are full of. Each rejected answer also has an even chance of carrying the same
markup and spacing noise as the accepted ones, so a metric cannot pick out the errors just by
finding the tidy ones.

`build_dataset` is given to you. Read it: every later cell scores what it makes.

In [ ]:
SUBJECTS = ["pump", "heater", "turbine", "conveyor", "compressor", "kiln",
            "boiler", "drone", "mixer", "generator", "cooler", "press"]

# dimension -> primary unit, alternative unit, how many alternative units make one primary
# unit, the values a template may draw (in the primary unit), and three phrasings of the
# attribute: the reference wording, a paraphrase, and a DIFFERENT attribute.
DIMENSIONS = {
    "power": dict(primary="kW", alt="W", factor=Fraction(1000),
                  values=["0.5", "0.75", "1.2", "1.5", "2.5", "3", "4.5", "7.5", "11", "15"],
                  forms=["runs at {q}", "draws {q}", "idles at {q}"]),
    "mass": dict(primary="kg", alt="g", factor=Fraction(1000),
                 values=["0.8", "1.5", "2.4", "12", "18", "25", "40", "64", "120"],
                 forms=["weighs {q}", "has a mass of {q}", "can lift {q}"]),
    "length": dict(primary="cm", alt="m", factor=Fraction(1, 100),
                   values=["45", "60", "80", "120", "150", "240"],
                   forms=["is {q} long", "has a length of {q}", "is {q} wide"]),
    "time": dict(primary="min", alt="s", factor=Fraction(60),
                 values=["2", "3", "5", "8", "12", "15", "45", "90"],
                 forms=["takes {q} to warm up", "needs {q} to warm up", "needs {q} to cool down"]),
}
SPELLED = {"W": "watts", "kW": "kilowatts", "g": "grams", "kg": "kilograms",
           "cm": "centimetres", "m": "metres", "s": "seconds", "min": "minutes"}
MARKUP = [("**", "**"), ("_", "_"), ("<b>", "</b>"), ("<strong>", "</strong>"), ("`", "`")]

ACCEPT_KINDS = ["markup", "spacing", "case", "reorder", "units", "paraphrase", "combined"]
REJECT_KINDS = ["digit", "swap", "negate", "wrong unit", "dropped fact", "wrong subject"]
FORMAT_KINDS = ("markup", "spacing", "case")      # the accepted kinds that only change formatting


def _decimal(x: Fraction, thousands: bool = False) -> str:
    """Exact decimal text of a Fraction whose denominator divides 1000 (optionally with commas)."""
    whole, frac = divmod(int(x * 1000), 1000)
    digits = f"{whole:,}" if thousands else str(whole)
    return digits if frac == 0 else digits + "." + f"{frac:03d}".rstrip("0")


def _value(num: str) -> Fraction:
    return Fraction(num.replace(",", ""))


def _sentence(subject, clauses, form=0, det="The", never=False, spelled=False,
              oxford=False, condition=False):
    """Render one sentence. `form` is one phrasing index, or one per clause."""
    forms = form if isinstance(form, (list, tuple)) else [form] * len(clauses)
    parts = []
    for c, f in zip(clauses, forms):
        unit = SPELLED[c["unit"]] if spelled else c["unit"]
        if spelled and _value(c["num"]) == 1:
            unit = unit.rstrip("s")
        parts.append(DIMENSIONS[c["dim"]]["forms"][f].format(q=f"{c['num']} {unit}"))
    if never:
        parts[0] = ("is never " + parts[0][3:]) if parts[0].startswith("is ") else "never " + parts[0]
    if condition:
        parts[0] += " only when idle"
    if len(parts) == 1:
        body = parts[0]
    else:
        joiner = ", and " if oxford and len(parts) > 2 else " and "
        body = ", ".join(parts[:-1]) + joiner + parts[-1]
    return f"{det} {subject} {body}."


def _in_alt_unit(c, thousands):
    spec = DIMENSIONS[c["dim"]]
    return dict(c, num=_decimal(_value(c["num"]) * spec["factor"], thousands), unit=spec["alt"])


def _markup(text, rng, subject):
    a, b = MARKUP[int(rng.integers(len(MARKUP)))]
    text = re.sub(rf"\b{subject}\b", a + subject + b, text, count=1)
    return "<p>" + text + "</p>" if rng.random() < 0.5 else text


def _spacing(text, rng):
    text = text.replace(" ", "  ", 1)                    # a doubled space after "The"
    text = re.sub(r"(\d) ", "\\1\u00a0", text)           # a non-breaking space before each unit
    text = re.sub(r",(?!\d)", " ,", text)                # a space before each comma
    if text.endswith("."):
        text = text[:-1] if rng.random() < 0.5 else text[:-1] + " ."
    return text


def _make_record(rng):
    subject = SUBJECTS[int(rng.integers(len(SUBJECTS)))]
    dims = list(DIMENSIONS)
    while True:
        n = int(rng.integers(2, 4))
        picked = [dims[i] for i in rng.permutation(len(dims))[:n]]
        clauses = [dict(dim=d, unit=DIMENSIONS[d]["primary"],
                        num=DIMENSIONS[d]["values"][int(rng.integers(len(DIMENSIONS[d]["values"])))])
                   for d in picked]
        if len({c["num"] for c in clauses}) == n:        # distinct numbers, so a swap changes something
            return subject, clauses


def _change_one_digit(c, rng):
    num = c["num"]
    positions = [i for i, ch in enumerate(num) if ch.isdigit()]
    while True:
        i = positions[int(rng.integers(len(positions)))]
        d = str((int(num[i]) + int(rng.integers(1, 10))) % 10)
        new = num[:i] + d + num[i + 1:]
        if not (i == 0 and d == "0") and _value(new) != _value(num):
            return dict(c, num=new)


def _items(subject, clauses, rng):
    """One reference, its accepted candidates and its rejected candidates, as (kind, text)."""
    n = len(clauses)
    reference = _sentence(subject, clauses)

    def reordered():
        while True:
            p = [int(i) for i in rng.permutation(n)]
            if p != list(range(n)):
                return [clauses[i] for i in p]

    def converted(cs):
        chosen = set(int(i) for i in rng.permutation(n)[:int(rng.integers(1, n + 1))])
        thousands = bool(rng.random() < 0.5)
        return [_in_alt_unit(c, thousands) if i in chosen else c for i, c in enumerate(cs)]

    accepted = [
        ("markup", _markup(reference, rng, subject)),
        ("spacing", _spacing(_sentence(subject, clauses, oxford=True), rng)),
        ("case", reference.lower()),
        ("reorder", _sentence(subject, reordered())),
        ("units", _sentence(subject, converted(clauses))),
        ("paraphrase", _sentence(subject, clauses, form=1, det="This")),
        ("combined", _markup(_sentence(subject, converted(reordered()), form=1, det="This"), rng, subject)),
    ]
    i, j = (int(x) for x in rng.permutation(n)[:2])
    swapped = list(clauses)
    swapped[i], swapped[j] = dict(clauses[i], num=clauses[j]["num"]), dict(clauses[j], num=clauses[i]["num"])
    wrong_unit = list(clauses)
    k = int(rng.integers(n))
    wrong_unit[k] = dict(clauses[k], unit=DIMENSIONS[clauses[k]["dim"]]["alt"])
    digit = list(clauses)
    k = int(rng.integers(n))
    digit[k] = _change_one_digit(clauses[k], rng)
    k = int(rng.integers(n))
    other = [s for s in SUBJECTS if s != subject]
    rejected = [
        ("digit", _sentence(subject, digit)),
        ("swap", _sentence(subject, swapped)),
        ("negate", _sentence(subject, clauses, never=True)),
        ("wrong unit", _sentence(subject, wrong_unit)),
        ("dropped fact", _sentence(subject, [c for q, c in enumerate(clauses) if q != k])),
        ("wrong subject", _sentence(other[int(rng.integers(len(other)))], clauses)),
    ]
    rejected = [(kind, _spacing(_markup(t, rng, subject), rng) if rng.random() < 0.5 else t)
                for kind, t in rejected]
    return reference, accepted, rejected


HELD_OUT_ACCEPT = ["spelled-out units"]
HELD_OUT_REJECT = ["added condition", "other attribute"]


def _held_out(subject, clauses):
    """Error types and a harmless rewrite that the rubric check in section 11 was NOT written for."""
    accepted = [("spelled-out units", _sentence(subject, clauses, spelled=True))]
    rejected = [("added condition", _sentence(subject, clauses, condition=True)),
                ("other attribute", _sentence(subject, clauses, form=[2] + [0] * (len(clauses) - 1)))]
    return accepted, rejected


def build_dataset(seed, n_records):
    """Given to you. `n_records` items, each a dict with the reference, its subject, its accepted
    and rejected candidates as (kind, text), and the held-out candidates for section 12."""
    rng = np.random.default_rng(seed)
    data = []
    for _ in range(n_records):
        subject, clauses = _make_record(rng)
        reference, accepted, rejected = _items(subject, clauses, rng)
        held_accepted, held_rejected = _held_out(subject, clauses)
        data.append(dict(ref=reference, subject=subject, accepted=accepted, rejected=rejected,
                         held_accepted=held_accepted, held_rejected=held_rejected))
    return data


DATA = build_dataset(SEED, 60)
print(f"{len(DATA)} items; per item {len(DATA[0]['accepted'])} accepted and "
      f"{len(DATA[0]['rejected'])} rejected candidates "
      f"({sum(len(d['accepted']) + len(d['rejected']) for d in DATA)} answers in all)\n")
_first = DATA[0]
print("reference:", repr(_first["ref"]))
for _kind, _text in _first["accepted"]:
    print(f"  accepted  {_kind:<13} {_text!r}")
for _kind, _text in _first["rejected"]:
    print(f"  REJECTED  {_kind:<13} {_text!r}")

## 3. Exercise 1 — `edit_distance()`

The edit distance between two sequences is the fewest single-element edits that turn one
into the other, where an edit is inserting one element, deleting one, or replacing one.
Every edit costs one. The classical way to compute it is dynamic programming: the distance
between the first `i` elements of `a` and the first `j` elements of `b` is built from the
three shorter prefixes, and you only ever need the previous row.

It must work on a string and on a list of words alike, so use only `len`, indexing and `==`
on the two arguments. A text metric normally divides the distance by the longer length to
get a similarity between 0 and 1; that wrapper is given below, so only the distance is
yours. OmniDocBench's overall score, as of 2 October 2026, includes one minus its text edit
distance in the same spirit (see `claims.yaml`).

<details><summary>💡 Hint 1 — what to think about</summary>

Turning an empty sequence into `b` takes `len(b)` insertions. For a longer prefix, the last
elements either match, which costs nothing extra, or they do not, and then the cheapest of
three earlier states plus one edit is the answer.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Keep a row of distances from the previous prefix of `a`, starting with the row for the empty
prefix. For each next element of `a`, build a new row whose first entry is the number of
elements consumed so far, and whose other entries take the smallest of: the entry above
plus one, the entry to the left plus one, and the entry diagonally up-left plus zero or one
depending on whether the two elements are equal. The last entry of the last row is the answer.
</details>

In [ ]:
def edit_distance(a, b):
    """Fewest single-element insertions, deletions and replacements that turn `a` into `b`.

    `a` and `b` are strings or lists; compare elements with `==`. Each edit costs 1. Swapping
    two neighbours is two edits, not one.

    Example:
        >>> edit_distance("kitten", "sitting")
        3
        >>> edit_distance(["the", "pump"], ["the", "big", "pump"])
        1

    Returns:
        an int, the edit distance.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def edit_similarity(a, b):
    """Given. 1 - edit_distance / the longer length, so identical is 1.0 and nothing shared is 0.0."""
    longest = max(len(a), len(b))
    return 1.0 if longest == 0 else 1.0 - edit_distance(a, b) / longest


def _check_edit_distance() -> None:
    assert edit_distance("kitten", "sitting") == 3, (
        f"got {edit_distance('kitten', 'sitting')} for kitten -> sitting; replacing, inserting "
        "and deleting one element each cost 1, so the answer is 3"
    )
    assert edit_distance("", "abc") == 3 and edit_distance("abc", "") == 3, (
        "against an empty sequence the distance is the other sequence's length: check the "
        "first row and the first column of your table"
    )
    assert edit_distance("ab", "ba") == 2, (
        f"got {edit_distance('ab', 'ba')} for ab -> ba; swapping two neighbours is two edits "
        "here (a replace and a replace), not one"
    )
    assert edit_distance(["the", "pump"], ["the", "big", "pump"]) == 1, (
        "it must work on lists of words: compare elements with ==, not characters"
    )
    assert edit_distance("same", "same") == 0 and edit_distance("", "") == 0
    assert edit_distance("flaw", "lawn") == edit_distance("lawn", "flaw") == 2, (
        "the distance is symmetric: a replace costs the same in both directions"
    )
    print("exercise 1 looks right")


_try("exercise 1", _check_edit_distance)

## 4. Exercise 2 — `bleu()`

BLEU was built for machine translation. For one candidate and one reference, each a list of
words, it does three things:

1. For each n-gram order `n` from 1 to `max_n`, it counts the candidate's n-grams and, for
   each distinct n-gram, credits it with at most as many matches as the reference contains
   (that cap is called **clipping**; without it, a candidate of one word repeated scores
   perfectly). The precision for order `n` is the clipped matches divided by the number of
   candidate n-grams.
2. It combines the precisions with a **geometric mean**: the exponential of the average of
   their logarithms. One zero precision makes the whole score zero.
3. It multiplies by a **brevity penalty**: 1 when the candidate is longer than the
   reference, otherwise `exp(1 - reference length / candidate length)`.

The original paper allows several references and a corpus of sentences, and notes that the
geometric mean is harsh when a precision vanishes, which it expected to be rare in a corpus
of reasonable size (`claims.yaml`). Here each answer is scored alone against ONE reference,
the way an item-level benchmark scores a page, so a vanishing precision is not rare. Neither
the paper nor this lesson says that is the best way to use BLEU; it is the way the
comparison is made. The rules for this lesson's version: an empty candidate scores 0.0, and
any order with no matching n-gram, including an order longer than the candidate, makes the
score 0.0.

One more difference from the paper: it computes the brevity penalty over the whole corpus and
says why, namely that applying it sentence by sentence would punish length deviations on short
sentences harshly (`claims.yaml`). This lesson applies it to every answer on its own anyway.
So treat this `bleu` as the lesson's own definition, written so that each step can be read and
tested here: its numbers are not claimed to match the output of any BLEU library, and they are
not comparable with published BLEU scores.

<details><summary>💡 Hint 1 — what to think about</summary>

`ngram_counts` (given, below) turns a list into a count of each n-gram. Clipping means that
for every n-gram in the candidate you take the smaller of two counts: how often the
candidate has it and how often the reference has it. Decide what happens before you take a
logarithm of a precision that could be zero.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Loop over the orders. For each, count candidate and reference n-grams, add up the clipped
matches and the candidate total, and stop with 0.0 if there are no matches. Otherwise add
the logarithm of the precision divided by the number of orders to a running sum. After the
loop compare the two lengths for the brevity penalty and return that times the exponential
of the running sum.
</details>

In [ ]:
def ngram_counts(seq, n):
    """Given. A Counter of the n-grams (as tuples) of a list or string, in order: for
    ["a", "b", "a"] and n = 2 it holds ("a", "b") once and ("b", "a") once."""
    return Counter(tuple(seq[i:i + n]) for i in range(len(seq) - n + 1))


def bleu(candidate, reference, max_n=4):
    """Sentence-level BLEU of one candidate against one reference, both lists of words.

    Clipped n-gram precision for each order 1..max_n, uniform geometric mean, times the brevity
    penalty (1 if the candidate is longer than the reference, else exp(1 - len(reference) /
    len(candidate))). An empty candidate, or any order with no matching n-gram (including an
    order longer than the candidate), scores 0.0.

    Example:
        >>> bleu(["the", "cat"], ["the", "cat"], max_n=2)
        1.0
        >>> round(bleu(["the", "cat", "sat"], ["the", "cat", "sat", "down"], max_n=1), 4)
        0.7165
        (every word matches, but the candidate is shorter, so the brevity penalty is exp(-1/3))

    Returns:
        a float between 0.0 and 1.0.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_bleu() -> None:
    same = bleu("the cat sat".split(), "the cat sat".split(), max_n=3)
    assert abs(same - 1.0) < 1e-9, f"a candidate identical to its reference must score 1.0, got {same}"
    got = bleu("the the the".split(), "the cat the".split(), max_n=1)
    assert abs(got - 2 / 3) < 1e-9, (
        f"got {got:.4f} for 'the the the' against 'the cat the' (one order); the reference has "
        "only two 'the', so only two matches count: clip each n-gram's matches to the reference's count"
    )
    got = bleu("the cat is".split(), "the cat is on the mat".split(), max_n=2)
    assert abs(got - math.exp(-1)) < 1e-9, (
        f"got {got:.4f}; every n-gram matches, so only the brevity penalty exp(1 - 6/3) is left. "
        "It applies when the candidate is SHORTER than the reference"
    )
    got = bleu("the cat sat on the mat".split(), "the cat is on the mat".split(), max_n=3)
    assert abs(got - 0.5) < 1e-9, (
        f"got {got:.4f}; the three precisions are 5/6, 3/5 and 1/4, and their GEOMETRIC mean is 0.5"
    )
    assert bleu("the cat sat".split(), "the cat is".split(), max_n=4) == 0.0, (
        "an order longer than the candidate (or with no match) makes the score 0.0, not an error"
    )
    assert bleu([], ["a"]) == 0.0
    print("exercise 2 looks right")


_try("exercise 2", _check_bleu)

## 5. Exercise 3 — `chrf()`

chrF is a character-level F-score. It was proposed as a stand-alone metric for machine
translation, and it does not depend on how the text is split into words. For one candidate
and one reference string (`claims.yaml` has the paper), this lesson's version:

1. Deletes ALL whitespace from both strings. The paper reports that counting spaces as
   characters gave no better agreement with human judgements and abandoned them.
2. For each character n-gram order `n` from 1 to `max_n` (the paper uses 6), computes the
   precision (clipped matches over candidate n-grams) and the recall (clipped matches over
   reference n-grams). An order for which either string has no n-grams, because it is
   shorter than `n`, is skipped.
3. Averages the precisions arithmetically over the orders that were kept, does the same
   with the recalls, and returns the F-score of those two averages:
   `(1 + beta^2) * P * R / (beta^2 * P + R)`. With `beta = 1` that is the harmonic mean;
   a larger `beta` weights recall more.

If no order is kept, or both averages are zero, the score is 0.0.

The paper says nothing about orders longer than a string, and the skip rule in step 2 is this
lesson's own choice (an implementation that counts such an order as a zero would print other
numbers). Like `bleu`, this `chrf` is the lesson's own definition: its numbers are not claimed to
match the output of any chrF library, and they are not comparable with published chrF scores.

<details><summary>💡 Hint 1 — what to think about</summary>

`ngram_counts` works on a string too. Precision and recall share their numerator, the
clipped matches, and differ only in what they divide by. The order in which you average and
apply the F formula matters: average first, then combine.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Remove whitespace with a join over a split. Loop over the orders, count both strings'
n-grams, skip the order if either total is zero, and otherwise append a precision and a
recall to two lists. If both lists are empty return 0.0; otherwise average each list, return
0.0 if both averages are zero, and apply the F formula to the two averages.
</details>

In [ ]:
def chrf(candidate, reference, max_n=6, beta=1.0):
    """Character n-gram F-score of `candidate` against `reference` (two strings).

    All whitespace is removed from both first. For each order 1..max_n whose n-grams exist in
    both strings, take the clipped-match precision and recall; average the precisions and the
    recalls over the orders kept, then return (1 + beta^2) * P * R / (beta^2 * P + R).
    Returns 0.0 if no order is kept or if both averages are zero.

    Example:
        >>> round(chrf("ab", "abcd", max_n=2), 4)
        0.5882
        (precision 1.0, recall about 0.4167 after averaging the two orders, F1 = 10/17)

    Returns:
        a float between 0.0 and 1.0.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_chrf() -> None:
    assert abs(chrf("pump runs", "pumpruns") - 1.0) < 1e-9, (
        "whitespace is deleted from both strings first, so these are identical and must score 1.0"
    )
    got = chrf("abc", "abd", max_n=2)
    assert abs(got - 7 / 12) < 1e-9, (
        f"got {got:.4f} for abc against abd; order 1 has precision and recall 2/3, order 2 has "
        "1/2 each, their arithmetic means are 7/12, and so is the F1"
    )
    got = chrf("ab", "abcd", max_n=2)
    assert abs(got - 10 / 17) < 1e-9, (
        f"got {got:.4f} for ab against abcd with beta 1; average the precisions and the recalls "
        "over the orders FIRST (precision 1, recall 5/12), then combine them: 10/17"
    )
    got = chrf("ab", "abcd", max_n=2, beta=2.0)
    assert abs(got - 25 / 53) < 1e-9, (
        f"got {got:.4f} with beta 2; recall is weighted more, so with these P and R the score is 25/53. "
        "Check which of precision and recall goes where in the formula"
    )
    assert abs(chrf("ab", "ab") - 1.0) < 1e-9, (
        "orders longer than the strings must be skipped, not counted as zero"
    )
    assert chrf("abc", "xyz") == 0.0 and chrf("", "abc") == 0.0, (
        "no overlap, or an empty string, scores 0.0 and must not divide by zero"
    )
    print("exercise 3 looks right")


_try("exercise 3", _check_chrf)

## 6. Score the first item, then look for the worst inversion

Now score every candidate of the first item against its reference, three ways. Each metric
returns a similarity: 1.0 means "the same as the reference". The BLEU here splits on
whitespace, which is the simplest thing a pipeline can do, and chrF and the edit similarity
work on the raw characters, markup and all.

Then, for each metric, the cell searches ALL the items for the worst inversion: the item in
which a candidate a reader would reject scores furthest above a candidate a reader would
accept. It prints both answers with their scores, so you can check by eye whether the
metric's opinion looks like a reader's.

In [ ]:
_TOO_SLOW: set[str] = set()


def _call(fn, *args):
    """Call into your code under a time limit, so a metric that never returns stops the demo.
    A function that has run out of time once is not called again: the later cells that need it
    say so at once instead of waiting for it again and again."""
    name = getattr(fn, "__name__", "a metric")
    if name in _TOO_SLOW:
        raise AssertionError(f"{name}() ran out of time in an earlier cell, so this cell is "
                             "skipped; make it faster, then restart and run from the top")
    try:
        with _time_limit(_CALL_SECONDS, name):
            return fn(*args)
    except AssertionError as exc:
        if str(exc).startswith(f"{name} was still running"):
            _TOO_SLOW.add(name)
        raise


RAW_METRICS = {
    "edit similarity": lambda text, item: _call(edit_similarity, text, item["ref"]),
    "BLEU": lambda text, item: _call(bleu, text.split(), item["ref"].split()),
    "chrF": lambda text, item: _call(chrf, text, item["ref"]),
}


def _scores(data, metric):
    """For each item: the metric's score for each accepted and each rejected candidate. Scores
    are rounded to nine places so that float noise cannot split a genuine tie."""
    return [([round(metric(t, item), 9) for _, t in item["accepted"]],
             [round(metric(t, item), 9) for _, t in item["rejected"]]) for item in data]


def _show_first_item() -> None:
    item = DATA[0]
    print("reference:", repr(item["ref"]))
    print(f"{'':9} {'kind':<13} " + " ".join(f"{name:>15}" for name in RAW_METRICS))
    for label, kind_text in (("accepted", item["accepted"]), ("REJECTED", item["rejected"])):
        for kind, text in kind_text:
            row = " ".join(f"{metric(text, item):15.3f}" for metric in RAW_METRICS.values())
            print(f"{label:9} {kind:<13} {row}")


def _worst_inversions(metrics, data=None) -> None:
    data = DATA if data is None else data
    for name, metric in metrics.items():
        best = None
        for item, (acc, rej) in zip(data, _scores(data, metric)):
            for ai, a in enumerate(acc):
                for ri, r in enumerate(rej):
                    if best is None or r - a > best[0]:
                        best = (r - a, item, ai, ri, a, r)
        gap, item, ai, ri, a, r = best
        print(f"{name}: a rejected answer outscores an accepted one by {gap:.3f}")
        print(f"  reference : {item['ref']!r}")
        print(f"  accepted  : {item['accepted'][ai][1]!r}  [{item['accepted'][ai][0]}]  scores {a:.3f}")
        print(f"  REJECTED  : {item['rejected'][ri][1]!r}  [{item['rejected'][ri][0]}]  scores {r:.3f}")


_try("first item scored", _show_first_item, needs=("exercise 1", "exercise 2", "exercise 3"))
print()
_try("worst inversions", lambda: _worst_inversions(RAW_METRICS),
     needs=("exercise 1", "exercise 2", "exercise 3"))

Read the three worst inversions. In each, would a human reading both answers next to the
reference prefer the one the metric prefers? Notice which kind of rejected answer wins, and
which kind of accepted answer loses. The next sections measure that over every item instead
of one hand-picked pair, and a pair picked as the worst of many is, by construction, not a
typical one: the tables that follow are the typical picture.

## 7. Exercise 4 — `misorder_rate()`: how often is the metric backwards?

A leaderboard orders systems by their average score. If a rejected answer scores above an
accepted one, a system that writes the rejected kind of answer is ranked above a system that
writes the accepted kind. So the measurement is a pair count: take every pair of one accepted
and one rejected answer for the SAME reference, and count the pairs the metric puts the wrong
way round.

`misorder_rate(accepted, rejected)` takes two sequences of scores (lists, tuples or numpy
arrays: the grader passes all three, so test for emptiness with `len`) and returns the share of pairs
`(a, r)`, one from each list, in which the rejected score `r` is higher than the accepted score
`a`, counting an exact tie as HALF a mistake (a metric that gives both answers the same score
is not telling them apart, which is a coin flip). "Exact" means `==` on the numbers you are given:
the scores that section 8 passes in have been rounded to nine decimal places first (see
`_scores`), so float noise cannot split a genuine tie, and two scores that differ at the ninth
place are not tied. So 0.0 means the metric always ranks the
accepted answers higher, 1.0 means it always ranks them lower, and 0.5 is no better than
chance. If either list is empty there are no pairs and the answer is 0.0.

<details><summary>💡 Hint 1 — what to think about</summary>

Every accepted score meets every rejected score exactly once, so there are as many pairs as
the two list lengths multiplied. Comparing every pair is one array comparison if you give
the two lists different shapes: one down a column, the other along a row.
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Turn the lists into arrays, one as a column and one as a row, and compare them elementwise
to get a table of booleans. Count the "rejected greater" cells and the "equal" cells
separately, add half the second count to the first, and divide by the number of pairs.
Handle the empty case before you divide.
</details>

In [ ]:
def misorder_rate(accepted, rejected):
    """Share of (accepted, rejected) score pairs in which the rejected score is higher, with
    each exact tie counting as half a mistake. 0.0 if either list is empty. The two arguments
    may be lists, tuples or numpy arrays.

    Example:
        >>> misorder_rate([0.9, 0.2], [0.5])
        0.5
        >>> misorder_rate([0.5], [0.5])
        0.5
        (the first pair is right, the second is wrong; the second case is a tie)

    Returns:
        a float between 0.0 and 1.0.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_misorder_rate() -> None:
    assert misorder_rate([0.9], [0.5]) == 0.0 and misorder_rate([0.5], [0.9]) == 1.0, (
        "a rejected answer that scores HIGHER than the accepted one is the mistake: it counts 1"
    )
    assert misorder_rate([0.5], [0.5]) == 0.5, (
        f"got {misorder_rate([0.5], [0.5])} for an exact tie; a tie counts as half a mistake"
    )
    assert misorder_rate([0.9, 0.2], [0.5]) == 0.5, "one pair of two is wrong: 0.5"
    got = misorder_rate([0.8, 0.4], [0.4, 0.6])
    assert abs(got - 0.375) < 1e-12, (
        f"got {got}; the four pairs score 0, 0, a tie (0.5) and 1, so the total is 1.5 over 4 pairs. "
        "Divide by the number of PAIRS, which is the two lengths multiplied"
    )
    assert misorder_rate([], [0.3]) == 0.0 and misorder_rate([0.3], []) == 0.0, (
        "with no pairs there is nothing to get wrong: return 0.0 rather than dividing by zero"
    )
    print("exercise 4 looks right")


_try("exercise 4", _check_misorder_rate)

## 8. The measurement: where is each metric backwards?

The cell below scores every candidate of every item with the three raw metrics and prints the
misordering rate three ways: by the kind of ACCEPTED answer (against all the rejected answers
of its own item), by the kind of REJECTED answer (against all the accepted ones), and over
all pairs. The summary rows average the accepted kinds that only change formatting, and the
rest.

Then it asks the question a team would ask next: "fine, pick a cutoff and call everything above
it accepted". `best_cutoff` (given) finds the cutoff that makes the fewest wrong calls on
this very data, which flatters the metric because no real team gets to choose the cutoff
after seeing the answers, and compares it with the dull rule "accept everything".

The mix of kinds is something this lesson chose: seven accepted kinds, of which three only
change formatting, and six rejected kinds. The overall row moves if you change the mix; the
per-kind rows do not, and are the ones to read.

In [ ]:
def best_cutoff(scores, labels):
    """Given. The cutoff (one of the scores, accept when score >= cutoff; the highest one on ties)
    that makes the fewest wrong calls, and how many wrong calls that is. `labels` are 1 for
    accepted answers and 0 for rejected ones."""
    best = None
    for cutoff in sorted(set(scores)):
        wrong = sum((s >= cutoff) != bool(l) for s, l in zip(scores, labels))
        if best is None or wrong <= best[1]:
            best = (cutoff, wrong)
    return best


def _print_misorder_tables(scored) -> None:
    names = list(scored)
    print("share of (accepted, rejected) pairs ranked the wrong way round (0 = never, 0.5 = chance)")
    print(f"{'ACCEPTED, by kind':<20}" + "".join(f"{n:>17}" for n in names))
    per_kind = {n: [np.mean([_call(misorder_rate, [acc[k]], rej) for acc, rej in scored[n]])
                    for k in range(len(ACCEPT_KINDS))] for n in names}
    for k, kind in enumerate(ACCEPT_KINDS):
        print(f"{kind:<20}" + "".join(f"{per_kind[n][k]:17.3f}" for n in names))
    fmt = [i for i, kind in enumerate(ACCEPT_KINDS) if kind in FORMAT_KINDS]
    rest = [i for i in range(len(ACCEPT_KINDS)) if i not in fmt]
    print(f"{'  formatting only':<20}" + "".join(f"{np.mean([per_kind[n][i] for i in fmt]):17.3f}" for n in names))
    print(f"{'  everything else':<20}" + "".join(f"{np.mean([per_kind[n][i] for i in rest]):17.3f}" for n in names))
    print(f"\n{'REJECTED, by kind':<20}" + "".join(f"{n:>17}" for n in names))
    for k, kind in enumerate(REJECT_KINDS):
        row = [np.mean([_call(misorder_rate, acc, [rej[k]]) for acc, rej in scored[n]]) for n in names]
        print(f"{kind:<20}" + "".join(f"{v:17.3f}" for v in row))
    print(f"{'ALL PAIRS':<20}" + "".join(
        f"{np.mean([_call(misorder_rate, acc, rej) for acc, rej in scored[n]]):17.3f}" for n in names))


def _print_cutoffs(scored) -> None:
    print("\nthe best single cutoff, chosen after seeing the answers (the most favourable case):")
    for n, rows in scored.items():
        flat = [(s, 1) for acc, _ in rows for s in acc] + [(s, 0) for _, rej in rows for s in rej]
        cutoff, wrong = best_cutoff([s for s, _ in flat], [l for _, l in flat])
        accept_everything = sum(1 for _, l in flat if l == 0)
        print(f"  {n:<16} cutoff {cutoff:.3f}: wrong on {wrong} of {len(flat)} answers; "
              f"accepting everything is wrong on {accept_everything}")


def _raw_measurement() -> None:
    scored = {name: _scores(DATA, metric) for name, metric in RAW_METRICS.items()}
    _print_misorder_tables(scored)
    _print_cutoffs(scored)
    zero_acc = sum(s == 0.0 for acc, _ in scored["BLEU"] for s in acc)
    zero_rej = sum(s == 0.0 for _, rej in scored["BLEU"] for s in rej)
    n_acc = sum(len(acc) for acc, _ in scored["BLEU"])
    n_rej = sum(len(rej) for _, rej in scored["BLEU"])
    print(f"\nBLEU scores of exactly 0.0: {zero_acc} of {n_acc} accepted answers, {zero_rej} of {n_rej} rejected ones")


_try("raw measurement", _raw_measurement,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"))

Read the accepted-kind rows first: which harmless rewrites does each metric punish, and are
the formatting-only ones punished as hard as the real rewordings? Then read the rejected-kind
rows: which errors does each metric let through? An edit distance of one character is what a
one-digit error costs, and it is also close to what a stray double space costs, so a metric
that only counts edits cannot tell a typo from a fact error.

Which kinds of accepted answer does chrF punish least, and why do you think that is? (Think
about what is left once the whitespace is gone and about what a reordered sentence keeps.)
And BLEU: the line under the table counts its exact zeros. Which kinds of answer are they?

## 9. Exercise 5 — `normalise()`: repair the formatting

The idea in the benchmark's January 2025 entry (normalised ground truths and predictions) is
the one to try first: put the reference and the candidate through the same normaliser before
comparing them, so that differences in formatting stop counting. `normalise(text)` must apply
these steps, in this order:

1. Unicode compatibility normalisation (NFKC), which among other things turns non-breaking
   spaces into ordinary ones and full-width letters and digits into plain ones.
2. Replace every HTML tag (anything from `<` to the next `>`) with a space.
3. Delete every `*`, `_` and backtick (Markdown emphasis and code marks).
4. Delete a comma with a digit before it and exactly three digits after it, not followed by a
   fourth digit: a thousands separator (`1,500` and `1,500kg`; not `1,50` or `1,5000`).
5. Collapse every run of whitespace to one space and trim the ends.
6. Delete whitespace before any of `, . ; : ! ?`.
7. Case-fold (so a German sharp s becomes `ss`, and step 8 also catches `, And `).
8. Turn `, and ` into ` and ` (the serial comma), then strip trailing full stops and spaces.

What a normaliser must NOT do is decide meaning: it makes every formatting variant of one
text the same string, and it has no idea that `1.5 kW` and `1500 W` are the same power. It
can still change meaning by accident, which the cell after the check measures.
`tokenize` (given) splits normalised text into numbers, words and punctuation marks, which
is what the BLEU of the repaired pipeline reads.

<details><summary>💡 Hint 1 — what to think about</summary>

The order matters. Tags become spaces before whitespace is collapsed, so that no two words
are glued together; spaces before punctuation can only be removed after the whitespace is
tidy. The standard library has a module for Unicode normal forms, and `re.sub` with a
lookbehind and a lookahead can express "a comma between digits".
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Write each step as one line that reassigns the text, in the order above. For thousands
separators: a digit behind the comma, three digits ahead and no fourth (a negative lookahead
inside the lookahead). Collapsing whitespace is a join over a split. Case-folding and
stripping the trailing characters are one string method each.
</details>

In [ ]:
def normalise(text):
    """Make formatting variants of the same text identical, following the eight steps above.

    Example:
        >>> normalise("The <b>pump</b>  draws 1,500 W ,  and weighs 12 kg.")
        'the pump draws 1500 w and weighs 12 kg'

    Returns:
        the normalised string.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def tokenize(text):
    """Given. Numbers (with a decimal part), words, and each punctuation mark as its own token."""
    return re.findall(r"\d+(?:\.\d+)?|\w+|[^\w\s]", text)


def _check_normalise() -> None:
    plain = "the pump runs at 1.5 kw and weighs 12 kg"
    assert normalise("The pump runs at 1.5 kW and weighs 12 kg.") == plain, (
        f"got {normalise('The pump runs at 1.5 kW and weighs 12 kg.')!r}; case-fold the text and "
        "strip the final full stop"
    )
    got = normalise("<p>The  <strong>pump</strong> runs at 1.5 kW , and weighs 12 kg .</p>")
    assert got == plain, (
        f"got {got!r}. Markup, a doubled space, a non-breaking space, a space before a comma and a "
        "serial comma must all disappear; do the tags first, then the whitespace, then the spaces "
        "before punctuation"
    )
    assert normalise("a<br>b") == "a b", (
        f"got {normalise('a<br>b')!r}; a tag becomes a SPACE, so the two words are not glued together"
    )
    assert normalise("draws 11,000 W") == "draws 11000 w", (
        "a comma between digits with exactly three digits after it is a thousands separator: delete it"
    )
    assert normalise("12 kg, 120 cm") == "12 kg, 120 cm", (
        "a comma that is NOT inside a number must stay: the number after it is a separate item"
    )
    assert normalise("１２ kg") == "12 kg", (
        "step 1 is NFKC normalisation, which also turns full-width digits into plain ones"
    )
    assert normalise("Done..") == "done", "strip every trailing full stop and space"
    assert normalise(normalise("<b>A</b> , and B .")) == normalise("<b>A</b> , and B ."), (
        "normalising twice must give the same result as normalising once"
    )
    print("exercise 5 looks right")


_try("exercise 5", _check_normalise)

### What the eight steps can break

Every step deletes or folds something. On this lesson's templates nothing that is deleted or
folded carries meaning, but that is a property of the templates, not of the steps. The cell
below (given) puts pairs of texts that MEAN different things through your `normalise` and
prints whether they come out as one string. A pair that does is a right answer and a wrong one
that every normalised metric in the next section would score as identical.

In [ ]:
_MEANING_PAIRS = [   # (the step that merges them, text, a text that means something else)
    ("case-fold", "runs at 5 MW", "runs at 5 mW"),          # megawatts and milliwatts
    ("tags", "draws <5 kW and >2 kW", "draws 2 kW"),         # a range, read as a tag
    ("NFKC", "rated 10\u00b2 W", "rated 102 W"),             # a superscript two (ten squared)
    ("marks", "a 2*3 m slab", "a 23 m slab"),                # an asterisk used as a times sign
    ("thousands", "weighs 1,500 kg", "weighs 1500 kg"),      # where a comma is the decimal mark
]


def _what_normalising_breaks() -> None:
    merged = 0
    for step, text, other in _MEANING_PAIRS:
        a, b = _call(normalise, text), _call(normalise, other)
        merged += a == b
        print(f"{step:<10} {text!r:<26} {other!r:<17} -> {a!r:<24} {b!r:<17} one string: {a == b}")
    print(f"{merged} of {len(_MEANING_PAIRS)} pairs with different meanings became one string")


_try("what normalising breaks", _what_normalising_breaks, needs=("exercise 5",))

None of these happens in this lesson's data, so the measurements below are not affected. A
normaliser for real answers has to be chosen for the text it will see (units are
case-sensitive, so a scorer of quantities should not fold their case; a document in a language
that writes `1,5` for one and a half needs another number rule) and tested with pairs like
these, just as the metrics are tested against answers whose verdict is known.

## 10. The measurement again, after normalising

Same items, same three metrics, but the reference and the candidate both go through
`normalise` first and BLEU reads `tokenize`d text. Compare each row with the raw table above.
You might expect only the formatting rows to move, since those answers differ from the
reference in formatting alone. Do they? Do the other rows move at all, and in which direction?
(Normalising changes the reference too, and the rejected answers carry formatting noise.)

In [ ]:
def _clean(text):
    return _call(normalise, text)


CLEAN_METRICS = {
    "edit similarity": lambda text, item: _call(edit_similarity, _clean(text), _clean(item["ref"])),
    "BLEU": lambda text, item: _call(bleu, tokenize(_clean(text)), tokenize(_clean(item["ref"]))),
    "chrF": lambda text, item: _call(chrf, _clean(text), _clean(item["ref"])),
}


def _clean_measurement() -> None:
    scored = {name: _scores(DATA, metric) for name, metric in CLEAN_METRICS.items()}
    print("AFTER normalising both sides (BLEU reads the tokenised text)\n")
    _print_misorder_tables(scored)
    _print_cutoffs(scored)
    raw = {name: _scores(DATA, metric) for name, metric in RAW_METRICS.items()}
    print("\nthe overall rate, before and after:")
    for n in CLEAN_METRICS:
        before = np.mean([_call(misorder_rate, a, r) for a, r in raw[n]])
        after = np.mean([_call(misorder_rate, a, r) for a, r in scored[n]])
        print(f"  {n:<16} {before:.3f} -> {after:.3f}")


_try("clean measurement", _clean_measurement,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 5"))

Normalising cannot do more than make formatting variants identical, so what is left over
is exactly what is NOT formatting: other orders, converted units, other words. No string
metric over one reference can score those as right, because to a string metric they are not.
Do you see the rejected kinds change at all? A normaliser also removes whatever formatting
noise had been making the real errors look different from the reference.

The remaining fix has to look at what the answer SAYS. For text whose facts are quantities,
that can be a rubric: a checklist the answer must satisfy, whatever its wording.

## 11. A rubric check, given: quantities, units and the subject

The check below is given, not an exercise. It pulls every number-with-a-unit out of a text,
converts it to the unit's base (so `1.5 kW` and `1,500 W` are the same exact fraction), and
requires that the candidate and the reference contain the same (dimension, value) pairs.
It also requires that the candidate does not contain a negation word and that it names the
reference's subject. It does not look at wording, order or markup.

It was written by looking at the list of error types above, which is the only list its author
had. Keep that in mind when you read its column.

In [ ]:
UNIT_TABLE = {   # unit -> (dimension, size of one unit in that dimension's base unit)
    "w": ("power", Fraction(1)), "kw": ("power", Fraction(1000)),
    "g": ("mass", Fraction(1)), "kg": ("mass", Fraction(1000)),
    "mm": ("length", Fraction(1, 1000)), "cm": ("length", Fraction(1, 100)),
    "m": ("length", Fraction(1)), "km": ("length", Fraction(1000)),
    "s": ("time", Fraction(1)), "min": ("time", Fraction(60)), "h": ("time", Fraction(3600)),
}
_QUANTITY = re.compile(r"(?<![\w.,])(\d+(?:,\d{3})*(?:\.\d+)?)\s*("
                       + "|".join(sorted(UNIT_TABLE, key=len, reverse=True)) + r")(?![A-Za-z])", re.I)
_NEGATION = re.compile(r"(?<![a-z])(not|never|no|cannot)(?![a-z])|n't")


def quantities(text):
    """Every number-with-a-unit in `text` as a sorted list of (dimension, exact value in base units)."""
    found = []
    for number, unit in _QUANTITY.findall(text.replace(" ", " ")):
        dimension, size = UNIT_TABLE[unit.lower()]
        found.append((dimension, Fraction(number.replace(",", "")) * size))
    return sorted(found)


def rubric_accepts(candidate, reference, subject):
    """True if the candidate has the reference's quantities, no negation, and the subject."""
    if quantities(candidate) != quantities(reference):
        return False
    lowered = candidate.casefold()
    if _NEGATION.search(lowered):
        return False
    return re.search(rf"(?<![a-z]){subject}(?![a-z])", lowered) is not None


RUBRIC_METRICS = {"rubric check": lambda text, item: 1.0 if rubric_accepts(text, item["ref"], item["subject"]) else 0.0}


def _show_quantities() -> None:
    item = DATA[0]
    print("reference     ", quantities(item["ref"]))
    for kind in ("units", "combined"):
        text = dict(item["accepted"])[kind]
        print(f"accepted {kind:<8}", quantities(text), "same as the reference:",
              quantities(text) == quantities(item["ref"]))
    for kind in ("digit", "swap", "wrong unit"):
        text = dict(item["rejected"])[kind]
        print(f"REJECTED {kind:<8}", quantities(text), "same as the reference:",
              quantities(text) == quantities(item["ref"]))


def _rubric_measurement() -> None:
    both = {**{n: _scores(DATA, m) for n, m in CLEAN_METRICS.items()},
            "rubric check": _scores(DATA, RUBRIC_METRICS["rubric check"])}
    _print_misorder_tables(both)
    wrong = sum(rubric_accepts(t, d["ref"], d["subject"]) != bool(label)
                for d in DATA for label, group in ((1, d["accepted"]), (0, d["rejected"]))
                for _, t in group)
    total = sum(len(d["accepted"]) + len(d["rejected"]) for d in DATA)
    print(f"\nthe rubric check calls {wrong} of these {total} answers wrongly")


_try("quantities", _show_quantities)
print()
_try("rubric measurement", _rubric_measurement,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 5"))

## 12. What the rubric check cannot see

The rubric column is clean because the answers it is scored on are the answers it was
written from: the generator and the checker share one list of error types, so the result says
how well the checker handles the cases its author thought of, and nothing about the others.
To find out, score it on candidates built to be outside that list:

* **spelled-out units**: the same facts written as `1.5 kilowatts`, which a reader accepts and
  a checker that only knows symbols does not recognise;
* **an added condition**: the quantities are unchanged but the sentence now says that its
  first fact holds only when the machine is idle, which changes the claim;
* **another attribute**: the first quantity is attached to a different attribute (idling
  at it, or being that wide) with the number and the unit unchanged.

These were chosen by this lesson's author to be blind spots of this particular checker, and
the lesson says so. The counts the cell prints are therefore zero BY CONSTRUCTION: they are not
a finding about how often such checkers fail on real answers, and they are not a rate. All
they show is that a checker's perfect score on its own list of cases is not a score on the
world. The cell prints, for each kind, how many of the answers the checker judges correctly,
with one example each.

In [ ]:
def _held_out_measurement() -> None:
    for label, key, kinds in ((1, "held_accepted", HELD_OUT_ACCEPT), (0, "held_rejected", HELD_OUT_REJECT)):
        for k, kind in enumerate(kinds):
            right = sum(rubric_accepts(d[key][k][1], d["ref"], d["subject"]) == bool(label) for d in DATA)
            example = DATA[0][key][k][1]
            print(f"{'a reader would accept' if label else 'a reader would REJECT':<22} {kind:<18} "
                  f"checker right on {right} of {len(DATA)}")
            print(f"    e.g. {example!r}  -> checker says "
                  f"{'accept' if rubric_accepts(example, DATA[0]['ref'], DATA[0]['subject']) else 'reject'}")


_try("held-out error types", _held_out_measurement)

So the rubric check is better than the raw metrics on the cases it was built for, and its
remaining errors are in the places nobody thought to look. A scorer revised only against the
failures someone has already seen can go this way. The way to see the next ones is to keep a
set of answers a reader would accept and answers a reader would reject, written by someone
other than the scorer's author, and to run every new scorer on them.

One more check, because a result that depends on one random draw is not a result: the same
experiment on three other seeds, with fewer items each so that it stays cheap.

In [ ]:
def _seed_spread() -> None:
    names = list(CLEAN_METRICS)
    print(f"{'seed':<10}{'items':>6}  " + "  ".join(f"{n + ' raw':>20}" for n in names)
          + "  " + "  ".join(f"{n + ' clean':>21}" for n in names) + "   rubric (listed / held out)")
    for seed in (SEED + 1, SEED + 2, SEED + 3):
        data = build_dataset(seed, 20)
        raw = [np.mean([_call(misorder_rate, a, r) for a, r in _scores(data, m)]) for m in RAW_METRICS.values()]
        clean = [np.mean([_call(misorder_rate, a, r) for a, r in _scores(data, m)]) for m in CLEAN_METRICS.values()]
        listed = sum(rubric_accepts(t, d["ref"], d["subject"]) != bool(lab)
                     for d in data for lab, g in ((1, d["accepted"]), (0, d["rejected"])) for _, t in g)
        held = sum(rubric_accepts(t, d["ref"], d["subject"]) != bool(lab)
                   for d in data for lab, g in ((1, d["held_accepted"]), (0, d["held_rejected"])) for _, t in g)
        n_held = sum(len(d["held_accepted"]) + len(d["held_rejected"]) for d in data)
        print(f"{seed:<10}{len(data):>6}  " + "  ".join(f"{v:20.3f}" for v in raw) + "  "
              + "  ".join(f"{v:21.3f}" for v in clean) + f"   wrong on {listed} / {held} of {n_held}")


_try("other seeds", _seed_spread,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 5"))

Compare the columns across seeds with the main run: do the same kinds of numbers appear? A
lesson-sized sample of twenty items will wobble more than sixty, and the ordering of the three
metrics against one another is the thing to check, not the third decimal.

## 13. Common mistakes

* **Treating a similarity score as a verdict.** A one-digit error and a stray space can cost
  the same number of edits. Look at which kinds of answer sit above which before choosing a
  cutoff, and never choose one after looking at the answers you will be graded on.
* **Normalising only one side.** The reference and the candidate must go through the same
  function, or the normaliser creates the differences it was meant to remove.
* **Believing a clean score on your own error list.** A checker written from a list of error
  types is perfect on that list by construction. Hold out kinds it was not written for.
* **Scoring with the arguments the wrong way round.** Edit distance is symmetric, and so is
  chrF at `beta` one. BLEU is not (its precisions divide by the CANDIDATE's n-grams, and its
  brevity penalty looks only at a short candidate), and neither is chrF with any other `beta`:
  the candidate and the reference play different roles. The cell below swaps them.
* **Reading BLEU's zero as "completely wrong".** It means one n-gram order had no match.
  The paper's own footnote expected that to be rare in a test corpus of reasonable size; one
  short answer scored alone is not a corpus.
* **Comparing scores from different versions of the scorer.** The benchmark in section 1 revised
  its matching code, so a number from before and a number from after are not on one scale.

In [ ]:
def _show_argument_order() -> None:
    item = DATA[0]
    text = dict(item["accepted"])["paraphrase"]
    print("candidate:", repr(text))
    print("reference:", repr(item["ref"]))
    for name, fn in (("edit similarity", lambda c, r: edit_similarity(c, r)),
                     ("BLEU", lambda c, r: bleu(c.split(), r.split())),
                     ("chrF (beta 1)", lambda c, r: chrf(c, r)),
                     ("chrF (beta 2)", lambda c, r: chrf(c, r, beta=2.0))):
        with _time_limit(_CALL_SECONDS, name):
            forward, backward = fn(text, item["ref"]), fn(item["ref"], text)
        print(f"  {name:<16} candidate vs reference {forward:.3f}   reference vs candidate {backward:.3f}")


_try("argument order", _show_argument_order, needs=("exercise 1", "exercise 2", "exercise 3"))

## 14. Self-check

1. A reference says `The pump runs at 1.5 kW.` Candidate X says `The pump runs at 15 kW.`
   Candidate Y says `Running at 1500 W, the pump...` in other words and in another order. A
   reader accepts Y and rejects X. Against this one reference, a plain edit-similarity metric
   will most likely:
   - (a) Score Y higher, because it is a correct answer
   - (b) Score both the same, because both differ by a number
   - (c) Reject X, because a changed number is a large edit
   - (d) Score X higher, because X is a single character away from the reference

2. BLEU uses a geometric mean of its n-gram precisions. For a short answer scored alone, the
   consequence is:
   - (a) One order with no matching n-gram makes the whole score zero
   - (b) Longer n-grams count for more, so word order is rewarded
   - (c) A candidate longer than the reference is penalised
   - (d) The score is the average of the precisions

3. You normalise the references but forget to normalise the candidates. What do you expect?
   - (a) No change, because both are strings
   - (b) Every score rises, because references are now simpler
   - (c) New differences between the two sides, so formatting variants of a correct answer can
     score worse than before
   - (d) The metric becomes symmetric

4. A rubric check scores perfectly on the answers its author generated. The right reading is:
   - (a) It will score perfectly on real answers
   - (b) It handles the error types its author thought of, and says nothing about the rest
   - (c) The generator is too easy and should be harder
   - (d) Rubric checks are better than metrics in every case

5. A benchmark's maintainers release a new version that changes how predictions are matched to
   the reference. A model's score on the old version and another model's score on the new one:
   - (a) Can be compared directly, because both are out of 100
   - (b) Can be compared if the models have the same size
   - (c) Can be compared after rounding
   - (d) Should not be compared until both are scored by the same version of the code

<details><summary>Answers</summary>

Answers come with this lesson's worked solution when you enrol on Synapsa.
</details>

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")

## What you built, and where it goes next

`edit_distance`, `bleu` and `chrf` are the three string metrics most benchmark scorers are
made of; `misorder_rate` is the measurement that says how often such a metric is backwards
on answers whose acceptability you know; `normalise` is the first repair, and the rubric
check is what the next one looks like. A scorer is a measuring instrument and deserves
the same tests as any other: known inputs, held-out cases, and a version number on its output.

What this lesson left out on purpose: learned metrics and embedding similarity (they need a
model, which this lesson does not have), several references per item, long documents with
reading-order and table structure, and any real human judgement. Each is a real part of
document-parsing evaluation, and none can be studied honestly until you can measure the
simple case, which is what you just did.

In [ ]:
# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    _ALL_CHECKS = (
        ("exercise 1", _check_edit_distance, ()),
        ("exercise 2", _check_bleu, ()),
        ("exercise 3", _check_chrf, ()),
        ("exercise 4", _check_misorder_rate, ()),
        ("exercise 5", _check_normalise, ()),
    )
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in _ALL_CHECKS:
            _try(_name, _check, needs=_needs)
    _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero. Inside a notebook kernel the
    # board above has already said so, in a line rather than a traceback.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.